# Mass photometry analysis: from `.mpr` files to masses

**How to use (no coding needed)**
1. Put your `.mpr` files in one folder: **one calibrant measurement** (e.g. MassFerence P1) and your **sample measurements**. Measure them on the same day with the same settings.
2. Change the settings in the grey cell below if needed. The defaults work for a folder containing a file with *Ladder* in its name.
3. In the menu, choose **Run → Run All Cells**.
4. The results appear below, and are saved as files in the `results` folder:

| File | What it contains |
|---|---|
| `all_reports.pdf` | every page below in one PDF |
| `summary.csv` | one row per peak for all samples (open in Excel) |
| `calibration/` | calibration report, `calibration.json` (reusable), peak residuals |
| `<sample>/` | `report.pdf`/`.png`, `peaks.csv`, `events.csv` (every particle with its mass), `warnings.txt` |

## 1. Settings

In [ ]:
DATA_FOLDER    = "."                  # folder with your .mpr files ("." = the folder of this notebook)
CALIBRANT_FILE = None                 # None = auto-detect (file/sample name contains 'ladder', 'calib', ...)
                                      #   or the file name, e.g. "002_Ladder.mpr"
CALIBRANT      = "MassFerence P1"     # calibrant name, or its known masses in kDa, e.g. [86, 172, 258, 344]
SAMPLE_FILES   = "*.mpr"              # which samples: "*.mpr" = all, "019_*.mpr", or ["a.mpr", "b.mpr"]
OUTPUT_FOLDER  = "results"

### Advanced settings (optional)

The defaults suit a Refeyn TwoMP with MassFerence P1. Change them only when there's a reason, for example peaks that are very close together, very large complexes, or a noisy sample. Everything used is saved to `results/settings_used.csv`.

In [ ]:
from mprfile import AnalysisSettings

SETTINGS = AnalysisSettings(
    mass_range=(40, 1500),   # kDa range in which peaks are searched and fitted
    bin_width=5,             # kDa histogram bin width
    min_fraction=0.03,       # ignore peaks with less than 3% of the events
    min_counts=20,           # ignore peaks with fewer than 20 events
    sigma_max=40,            # kDa: maximum peak width (σ)
    max_fit_error=None,      # e.g. 0.2 to discard poorly fitted particles
    background=False,        # True: fit a broad background under the peaks
    plot_max=None,           # kDa: right edge of the plots (None = automatic)
)

## 2. Run the analysis

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mprfile import analyze
from mprfile.report import plot_calibration, plot_sample

folder = Path(DATA_FOLDER)
samples = [str(folder / s) for s in ([SAMPLE_FILES] if isinstance(SAMPLE_FILES, str) else SAMPLE_FILES)]
cal_file = str(folder / CALIBRANT_FILE) if CALIBRANT_FILE else None

results = analyze(samples, calibrant_file=cal_file, calibrant=CALIBRANT,
                  out=folder / OUTPUT_FOLDER, settings=SETTINGS)

## 3. Calibration

Check that the **black labels** sit on the calibrant peaks and that there are **no warnings**. For MassFerence P1, the higher ladder peaks (orange) are an independent check: they weren't used in the fit, so if they land within a few % of 430/516/602 kDa, the calibration is also linear above 344 kDa.

In [ ]:
fig = plot_calibration(results.calibration); plt.show()

## 4. Results per sample

In [ ]:
for r in results.samples:
    fig = plot_sample(r); plt.show()

## 5. Summary table (all samples)

Also saved as `summary.csv`. **mass ± fit** is the uncertainty of the peak position from the fit, **σ** is the peak width, and **%** is the share of binding events in the fitted mass range.

In [ ]:
s = results.summary
if len(s) and "mass_kDa" in s:
    view = s[["sample", "file", "peak", "mass_kDa", "mass_err_kDa", "sigma_kDa", "counts", "percent", "notes"]].copy()
    view.columns = ["sample", "file", "peak", "mass (kDa)", "± fit (kDa)", "σ (kDa)", "events", "%", "notes"]
    display(view.style.format({"mass (kDa)": "{:.0f}", "± fit (kDa)": "{:.1f}", "σ (kDa)": "{:.0f}",
                               "events": "{:.0f}", "%": "{:.1f}"}).hide(axis="index"))
else:
    print("No peaks found.")

## 6. Compare samples

Normalised mass distributions of all samples overlaid, useful for dilution series or before/after comparisons. The x-axis range follows the advanced settings.

In [ ]:
import numpy as np
if results.samples:
    edges = np.arange(0, (SETTINGS.plot_max or 1000) + SETTINGS.bin_width, SETTINGS.bin_width)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    for r in results.samples:
        m = r.events.mass[r.events.mass > 0]
        h, _ = np.histogram(m, edges)
        ax.stairs(h / max(h.sum(), 1) * 100, edges, lw=1.4, label=f"{r.info.get('sample')} ({r.name})")
    ax.set(xlabel="Mass (kDa)", ylabel=f"% of binding events per {SETTINGS.bin_width:g} kDa",
           title="Binding events, normalised")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    plt.show()

## How to read the results

- **Binding ↑ / unbinding ↓.** Mass photometry records particles that land on the glass (binding, plotted up) and particles that leave it (unbinding, plotted down, mirrored). Masses and percentages are calculated from **binding** events only. An unbinding peak at the same mass is normal.
- **Orange peak labels / notes** mark results to interpret with care:
  - *below/above calibrated range*: the mass lies outside the calibrant peaks, so the calibration line is extrapolated.
  - *width hit the limit*: this isn't one clean species (a broad or smeared population), so the mass and % are approximate.
  - *cut off by lower mass limit*: part of the peak lies below the fitted range (default 40 kDa). Very small particles approach the detection limit.
- **%** is the share of binding events in the fitted mass range. Events between or below peaks aren't assigned to any peak, so the percentages don't have to add up to 100.
- **Calibration warnings** (for example a different instrument, different camera settings, or a calibrant measured on another day) mean the calibration may not apply to that sample.
- **For a paper or report,** give each peak's mass ± σ and its number of events, and name the calibrant and the calibration file.

This analysis uses the particles AcquireMP detected and fitted, converted to mass with a calibration fitted independently of AcquireMP. For a spot check, compare a sample with DiscoverMP.

## Expert mode

Every step is available from Python, for example to reuse a calibration or change one sample's settings:

```python
from mprfile import calibrate, analyze_sample, AnalysisSettings, Calibration
cal = calibrate("002_Ladder.mpr", "MassFerence P1")        # or Calibration.load("results/calibration/calibration.json")
print(cal.report())
r = analyze_sample("019_252_50nM.mpr", cal, AnalysisSettings(bin_width=4, background=True))
r.peaks, r.events, r.qc, r.warnings
```

Or from the command line (Anaconda Prompt, with the `mpr` env active):

```
mpr-analyze                                              # all *.mpr in this folder, auto-detected calibrant
mpr-analyze data\*.mpr -c data\002_Ladder.mpr -o results
mpr-analyze new_samples\*.mpr --calibration results\calibration\calibration.json
mpr-analyze --help                                       # all options
```

`mpr_tutorial.ipynb` covers the raw data: movies, the ratiometric frames, the PSF, kinetics and more.